In [1]:
import duckdb
from pathlib import Path
import polars as pl 
import sys

# Adds the parent directory of the notebook (i.e., project root) to sys.path
sys.path.append(str(Path().resolve().parent))

import src.transform_utils as tu


In [2]:
db_file = Path("../data/NSQIP_P_Clean.duckdb")

print(f"Using DuckDB database: {db_file}")

if db_file.exists():
    size_mb = db_file.stat().st_size / (1024 * 1024 * 1024)
    print(f"File exists. Size: {size_mb:.2f} GB")
else:
    print("File does not exist.")
    
# get table name (only one table in the database)
with duckdb.connect(db_file) as con:
    tables = con.execute("SHOW TABLES").fetchall()
    print(f"Tables in the database: {tables}")
    table_name = tables[0][0] if tables else None
    print(f"Table name: {table_name}")

Using DuckDB database: ..\data\NSQIP_P_Clean.duckdb
File exists. Size: 0.32 GB
Tables in the database: [('all_data_table',)]
Table name: all_data_table


In [ ]:
# cols that can be changed to categorical
to_categorical = ['ADMYR', 'CASEID', 'CONCPT10', 'CONCPT6',
                  'CONCPT7', 'CONCPT8', 'CONCPT9', 'CPT', 'HDISDT',
                  'OTHERCPT10', 'OTHERCPT8', 'OTHERCPT9', 'PUFYEAR']

tu.cast_column_in_place(
    db_path=db_file,
    table_name=table_name,
    columns_to_cast=to_categorical,
)

Column 'ADMYR' casted to STRING in table 'all_data_table'.
Column 'CASEID' casted to STRING in table 'all_data_table'.
Column 'CONCPT10' casted to STRING in table 'all_data_table'.
Column 'CONCPT6' casted to STRING in table 'all_data_table'.
Column 'CONCPT7' casted to STRING in table 'all_data_table'.
Column 'CONCPT8' casted to STRING in table 'all_data_table'.
Column 'CONCPT9' casted to STRING in table 'all_data_table'.
Column 'CPT' casted to STRING in table 'all_data_table'.
Error casting column 'HDISDTOTHERCPT10' in table 'all_data_table': Binder Error: Referenced column "HDISDTOTHERCPT10" not found in FROM clause!
Candidate bindings: "__tmp_HDISDTOTHERCPT10", "HDISDT", "OTHERCPT10", "OTHERCPT1", "OTHERPROC10"

LINE 1: UPDATE all_data_table SET __tmp_HDISDTOTHERCPT10 = CAST(HDISDTOTHERCPT10 AS STRING);
                                                                ^
Column 'OTHERCPT8' casted to STRING in table 'all_data_table'.
Column 'OTHERCPT9' casted to STRING in table 'all_data

In [4]:
# columns to make CPT list from
cpt_cols = ['CPT', 'CONCPT1', 'CONCPT2', 'CONCPT3', 'CONCPT4', 
            'CONCPT5', 'CONCPT6', 'CONCPT7', 'CONCPT8', 'CONCPT9',
            'CONCPT10', 'OTHERCPT1', 'OTHERCPT2', 'OTHERCPT3',
            'OTHERCPT4', 'OTHERCPT5', 'OTHERCPT6', 'OTHERCPT7',
            'OTHERCPT8', 'OTHERCPT9', 'OTHERCPT10']

tu.add_combined_cpt_column(
    db_path=db_file,
    table_name=table_name,
    cpt_columns=cpt_cols,
)

BinderException: Binder Error: Cannot create a list of types VARCHAR and INTEGER - an explicit cast is required

LINE 3: ..., OTHERCPT5, OTHERCPT6, OTHERCPT7, OTHERCPT8, OTHERCPT9, OTHERCPT10]
                                                                    ^

In [ ]:
# Sum of RVU cols for initial surgery
rvu_cols = ['CONWRVU1', 'CONWRVU2', 'CONWRVU3', 'CONWRVU4',
            'CONWRVU5', 'CONWRVU6', 'CONWRVU7', 'CONWRVU8',
            'CONWRVU9', 'CONWRVU10', 'OTHERWRVU1', 'OTHERWRVU2',
            'OTHERWRVU3', 'OTHERWRVU4', 'OTHERWRVU5', 'OTHERWRVU6',
            'OTHERWRVU7', 'OTHERWRVU8', 'OTHERWRVU9', 'OTHERWRVU10',
            'WORKRVU']

tu.add_total_rvu(
    db_path=db_file,
    table_name=table_name,
    rvu_columns=rvu_cols,
)

In [ ]:
# 'ANESTECH' has no 2013 values and 'ANESTHES' has only 2013 values
# need to combine them into one column

# update this to take the columns as a list

tu.combine_race_cols(
    db_path=db_file,
    table_name=table_name,
)

In [ ]:
# there are 'BIRTH_HGT' and 'BIRTH_HGT_UNIT' columns 
# should find a way to standardize to a single measurement unit

# maybe also with 'BIRTH_WGT_UNIT'

# and 'HEAD_CIRC'

In [ ]:
tu.add_free_flap_flags(
    db_path=db_file,
    table_name=table_name,
)